### Spatial niches: ligand-receptor pairs, gradients and biological programs

Follows `squid01_analyze_visium_HE_data.ipynb` (mouse brain, Visium H&E, `visium_hne.h5ad`).

For a central spot (or the core of a region):
1. map the k ≤ 20 nearest spots (Visium hex grid: 6 + 12 spots in rings 1-2),
2. score every ligand-receptor (LR) pair in both directions, **center → neighbors** and **neighbors → center**, against 2,000 random niches of the same tissue (z, empirical p, FDR),
3. measure ligand and receptor **gradients** along 3-6 collinear spots, from the center towards the best partner spot,
4. infer **biological programs** (50 functional programs, Reactome-summarised) from niche program activity + programs containing the significant L/R genes.

Caveat: a Visium spot is 55 µm (≈1-10 cells), so these are spot-level niches; gradients over 3-6 points are descriptive.

### Visium Hexagonal Grid

The standard 10x Genomics Visium platform (such as Visium v1 and v2) arranges its gene expression capture spots in a hexagonal (honeycomb) grid, whereas newer platforms like Visium HD use a continuous square grid.

1. [Visium Data Preprocessing](https://lmweber.org/Visium-data-preprocessing/introduction.html)
2. [OSTA (Biocondutor): Introduction to sequencing-based ST & 10x Genomics Visium Platforms](https://lmweber.org/OSTA/pages/seq-introduction.html)
  - [CCC](https://lmweber.org/OSTA/pages/img-cell-cell-communication.html)
3. [paper: Integrating gene expression and imaging data across Visium capture areas with visiumStitched](https://pmc.ncbi.nlm.nih.gov/articles/PMC11326277/)


#### Key Specifications of the Visium Hexagonal Grid

* Spot Diameter: 55 µm [1] 
* Center-to-Center Spacing: 100 µm hexagonal spacing [1] 
* Array Dimensions: Typically 6.5 mm × 6.5 mm (or larger 11 mm capture areas on supported CytAssist setups) containing around 5,000 barcoded spots. [1, 3] 
* Neighbor Structure: Each spot on a standard hexagonal grid has up to 6 immediate neighbors, which downstream spatial statistics and clustering tools (such as [BayesSpace](https://www.bioconductor.org/packages/release/bioc/html/BayesSpace.html) or [Seurat](https://satijalab.org/seurat/)) use as a proxy for physical distance. [3, 4, 5] 
* Cell Overlap: Varies by tissue type, averaging roughly 0 to 10 cells per spot in human brain tissue or up to ~50 cells per spot in denser tissues. [2] 

If you are working on a specific analysis pipeline, let me know:

* Are you using standard Visium (hexagonal) or Visium HD (square)?
* What tool or framework (e.g., Seurat, Scanpy, or Bioconductor) are you using for downstream clustering?


4. [pachterlab](https://pachterlab.github.io/voyager/articles/vig2_visium.html)
5. [bioconductor](https://www.bioconductor.org/packages/devel/bioc/html/visiumStitched.html)


In [1]:
import numpy as np
import pandas as pd
import os, sys

import anndata as ad
import scanpy as sc
import squidpy as sq
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/gwas/")
ROOT_SRC = ROOT0 / "src"
sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import *

import matplotlib.pyplot as plt

root0 = Path('/home/flavio/uv/squidpy/')
root_data = root0 / 'data'
root_figure = root0 / 'figures'
root_result = root0 / 'results'


root_colab = Path('/home/flavio/uv/colab')
root_reactome = root_colab / 'reactome'

# folder with niche_lr.py, programs.py, lr_mouseconsensus.csv
ROOT_SRC = root0 / 'src' 
sys.path.append(str(ROOT_SRC))

import programs.niche_lr as nl
import programs.programs as pg

In [2]:
filename_vis = root_data / "visium_hne.h5ad"
adata = ad.read_h5ad(filename_vis)
adata

AnnData object with n_obs × n_vars = 2688 × 18078
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'leiden', 'cluster', 'features_cluster'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_colors', 'features_cluster_colors', 'hvg', 'leiden', 'leiden_colors', 'liana_res', 'neighbors', 'pca', 'rank_genes_groups', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'features', 'features_summary_scale1.0', 'features_summary_scale2.0', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 

### Expression layer

The niche scores need **log-normalised, non-negative** values. In this object `adata.X` is log-normalised and `.raw` holds counts, so `layer=None` (use `X`).

In [3]:
X = adata.X
print(type(X), X.shape, "min", X.min(), "max", X.max())

coords = adata.obsm["spatial"]
spacing = nl.spot_spacing(coords)          # ~137 px centre-to-centre
um_per_px = 100 / spacing                  # Visium: 100 um centre-to-centre
print(f"spot spacing = {spacing:.1f} px  ->  {um_per_px:.3f} um/px")

<class 'scipy.sparse._csr.csr_matrix'> (2688, 18078) min 0.0 max 8.859275
spot spacing = 137.0 px  ->  0.730 um/px


### Programs (human catalogue -> mouse symbols)

`to_mouse` capitalises human symbols and applies explicit orthologs (HLA -> H2, FCGR3A -> Fcgr4, ...); genes absent from the data are dropped and poorly covered programs are reported.

In [4]:
programs_m = pg.to_mouse(pg.PROGRAMS, adata.var_names)
states_m   = pg.to_mouse(pg.CELL_STATES, adata.var_names)

df_prog = pd.DataFrame([(c, p, len(g)) for c, d in programs_m.items() for p, g in d.items()],
                       columns=["category", "program", "n_genes_in_data"])
print(df_prog.shape)
df_prog.sort_values("n_genes_in_data").head(10)

programs with <50% genes found in data:
        category    program  n_human  n_mouse  n_in_data
epithelial_PDAC  classical        7        6          2
epithelial_PDAC basal_like        7        7          2
(51, 3)


,category,program,n_genes_in_data
1,cell_death,apoptosis_extrinsic,10
3,cell_death,pyroptosis_inflammasome,11
19,metabolism,iron_metabolism,11
2,cell_death,necroptosis,12
46,immune,T_cell_exhaustion_checkpoint,12
47,immune,humoral_B_cell,12
34,signaling,Hedgehog_signaling,12
11,stress,senescence_SASP,13
41,immune,complement,13
5,cell_death,necrosis_DAMP,13


### Reactome: every pathway -> program (or background class)

`reactome_pathways.tsv` has names only (no genes), so this is the name-based summary of all Reactome pathways into the 50 programs.
Pathways that are disease variants, pathogens or drugs are *excluded*; housekeeping biology goes to *context* classes (transcription, translation, transport, GPCR, ...).
With `ReactomePathways.gmt` (gene members) use `pg.expand_with_reactome(pg.load_gmt(path))` to build the complete gene sets.

In [5]:
filename_reac = root_reactome / "reactome_pathways.tsv"

df_reac = pg.load_reactome_tsv(filename_reac, species="Mus musculus")
df_cls  = pg.classify_reactome(df_reac)
df_sum  = pg.summarize_reactome(df_cls)

print(df_cls.drop_duplicates("pathway").level.value_counts())

fname = "reactome_mouse_pathway_to_program.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_cls, fname, root_result, verbose=True)

df_sum[df_sum.level == "program"].head(6)

level
program         1388
context          384
exclude           45
unclassified       5
Name: count, dtype: int64


,level,category,label,n_pathways
11,program,signaling,RTK_MAPK,198
12,program,metabolism,fatty_acid_lipid,135
13,program,neural,synaptic_neurotransmission,108
14,program,stress,DNA_damage_response,91
15,program,proliferation,G2_M_mitosis,78
16,program,immune,inflammation_TNF_NFkB,74


### Choose the center

Options: `barcode=...`, `xy=(x, y)`, `mask=<bool array or obs column>`, `program="synaptic_neurotransmission"` (hotspot), or one center per anatomical cluster (`centers_by_group`: the most interior spot of each cluster).

In [6]:
centers = nl.centers_by_group(adata=adata, groupby="cluster")
centers

{'Cortex_2': 2135,
 'Cortex_5': 127,
 'Thalamus_2': 2400,
 'Hypothalamus_2': 1828,
 'Cortex_4': 700,
 'Fiber_tract': 1491,
 'Lateral_ventricle': 1982,
 'Cortex_1': 1367,
 'Hypothalamus_1': 1284,
 'Striatum': 2041,
 'Hippocampus': 2583,
 'Pyramidal_layer': 725,
 'Thalamus_1': 1631,
 'Cortex_3': 1361,
 'Pyramidal_layer_dentate_gyrus': 2588}

In [7]:
cluster_name = "Hippocampus"
center = centers[cluster_name]

center

2583

In [8]:
# alternatives:
# center = nl.select_center(adata, program="synaptic_neurotransmission", programs=programs_m)
# center = nl.select_center(adata, barcode="AAACAAGTATCTCCCA-1")

lr_m = nl.load_lr("mouseconsensus")
print(lr_m.shape, adata.obs_names[center], adata.obs.loc[adata.obs_names[center], "cluster"])

(3989, 2) TTCCTCGAGGGTGTCT-1 Hippocampus


### Run the niche analysis

In [10]:
len(programs_m), list(programs_m.keys())

(9,
 ['cell_death',
  'stress',
  'proliferation',
  'metabolism',
  'trafficking',
  'tissue_remodeling',
  'signaling',
  'immune',
  'neural'])

In [15]:
# background: whole tissue (None) -> "what is special about this niche";
#             other clusters        -> "this region vs the rest"
bg_mask = (adata.obs["cluster"] != cluster_name).values
# bg_mask = None

res = nl.analyze_niche(
    adata, center, k=20,
    center_pool=6,         # center + ring 1 pooled into a central region (damps single-spot noise);
                           # partners = the remaining 14 spots (ring 2 + 2 of ring 3). 0 = single spot
    lr=lr_m, programs=programs_m,
    bg_mask=bg_mask,
    layer=None,            # adata.X is log-normalised
    fdr=0.1,
    n_top_gradients=15,
    n_line_cells=6,        # 3-6 collinear spots
    um_per_unit=um_per_px, # gradients per um
    n_background=2000,
)
df_neig = res.neighbors.copy()
print(df_neig.shape)
df_neig.head(8)

(20, 7)


,idx,barcode,dx,dy,dist,ring,angle_deg
0,1346,CTTGTACTTGTTGACT-1,137,0,137.000000,1,0.000000
1,1380,GAACACACATCAACCA-1,69,119,137.557261,1,59.893473
2,855,CCAAACAGAACCCTCG-1,-69,119,137.557261,1,120.106527
3,1636,GCTAATACCGAATGCC-1,-138,0,138.000000,1,180.000000
4,2564,TTCAACGACCCGACCG-1,-69,-120,138.423264,1,240.101098
5,683,CAACGACCCGTTTACA-1,69,-120,138.423264,1,299.898902
6,383,AGCATCGTCGATAATT-1,206,119,237.901240,2,30.013733
7,2325,TCTCTAATAGCTGGTA-1,206,-120,238.403020,2,329.778162


### Neighbours: cluster composition

In [ ]:
nb["cluster"] = adata.obs["cluster"].values[nb.idx]
nb.groupby(["ring", "cluster"], observed=True).size().unstack(fill_value=0)

### Top LR pairs (both directions)

`center->neighbors`: ligand in the center, receptor in the neighbours. `neighbors->center`: the reverse.
`z` = niche score vs. random niches. `fdr` uses the empirical p, switching to the Gaussian tail of z when fewer than 5 background niches reach the observed score (the empirical p has a floor of 1/(n_background+1)); `fdr_emp` is the purely empirical version.

In [ ]:
cols = ["ligand", "receptor", "direction", "score", "z", "fdr", "fdr_emp",
        "ligand_sender", "receptor_receiver", "ligand_programs", "receptor_programs"]

df_lr = res.lr

fname = f"niche_lr_{cluster_name}.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_lr, fname, root_result, verbose=True)

df_lr[cols].head(25)

### Inferred biological functions

`z` = niche program activity vs. random niches; `lr_support` = Σ -log10(p) of significant LRs whose L or R is in the program; `score` combines both ranks.

In [ ]:
df_fun = res.functions

fname = f"niche_functions_{cluster_name}.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_fun, fname, root_result, verbose=True)

df_fun[["category", "program", "z", "percentile", "lr_support", "n_lr", "top_lr", "score"]].head(15)

### Cell states in the niche (identity markers, activity only)

In [ ]:
df_states = nl.niche_program_activity(adata, center, res.neighbors, states_m, n_background=2000, bg_mask=bg_mask)
df_states.head(10)

### Gradients along collinear spots

A slope counts as a trend only if |Spearman ρ| ≥ 0.6 (`pattern`: source→sink, co-decreasing, ligand/receptor gradient only, flat).
`consistent_with_direction`: center→neighbors → ligand highest at the center and decreasing, receptor expressed downstream; neighbors→center → receptor highest at the center, ligand rising towards the sender.

In [ ]:
df_grad = res.gradients
df_grad[["key", "n_cells", "L_slope", "L_rho", "R_slope", "R_rho", "LR_corr",
         "pattern", "consistent_with_direction", "z", "fdr"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
nl.plot_niche(adata, res, ax=ax)

filename = root_figure / f"niche_map_{cluster_name}.png"
if not filename.exists():
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()

In [ ]:
keys = df_grad.sort_values(["consistent_with_direction", "z"], ascending=False).key.head(6).tolist()
fig = nl.plot_gradients(res, keys)

filename = root_figure / f"niche_gradients_{cluster_name}.png"
if not filename.exists():
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()

### Cross-check with LIANA+ bivariate (from squid01)

Local cosine score of the same LR at the center and its neighbours, and global rank by Moran's R.

In [ ]:
filename_lr = root_data / "visium_hne_liana_bivariate.h5ad"
if filename_lr.exists():
    lrdata = ad.read_h5ad(filename_lr)
    df_chk = nl.liana_local_check(lrdata, res, n=25)
    display(df_chk)

### All clusters: program x niche matrix

In [ ]:
fname = "niche_functions_by_cluster.tsv"
force = False

if (root_result / fname).exists() and not force:
    df_mat = pdreadcsv(fname, root_result).set_index("program")
else:
    df_mat, results = nl.scan_niches(adata, centers, value="z", exclude_own_group="cluster",
                                     center_pool=6, lr=lr_m, programs=programs_m,
                                     um_per_unit=um_per_px, n_background=1000, n_top_gradients=10)
    pdwritecsv(df_mat.reset_index(), fname, root_result, verbose=True)

df_mat.round(1).head().T

In [ ]:
import seaborn as sns

top = df_mat.loc[df_mat.abs().max(1).sort_values(ascending=False).index[:30]]
g = sns.clustermap(top, cmap="coolwarm", center=0, figsize=(12, 10), vmin=-5, vmax=5,
                   cbar_kws={"label": "niche activity z"})

filename = root_figure / "niche_programs_by_cluster.png"
if not filename.exists():
    g.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()